# 🔥 Notebook 5: QLoRA Fine-Tuning with Unsloth
> ⚠️ **Run this notebook on Kaggle (T4 x2) or Google Colab (A100). Do NOT run locally unless you have a 16GB+ GPU.**

**Goal:** Fine-tune Qwen 2.5 2B with QLoRA on Gujarati healthcare JSONL data.

**Output:** `models/qwen_gu_health_lora/` (LoRA adapter weights)

## 0. Install Unsloth (Run First on Kaggle/Colab)

In [ ]:
# KAGGLE: Run this cell first, then restart kernel
import subprocess
import sys


def install_unsloth():
    # Detect environment
    try:
        import google.colab

        env = "colab"
    except ImportError:
        env = "kaggle"

    print(f"Environment: {env}")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
            "xformers",
            "trl",
            "peft",
            "accelerate",
            "bitsandbytes",
        ]
    )


install_unsloth()
print("✅ Unsloth installed. Restart kernel if this is first run.")

## 1. GPU Verification

In [ ]:
import torch

assert torch.cuda.is_available(), "❌ No GPU found! This notebook MUST run on GPU."

num_gpus = torch.cuda.device_count()
total_vram = (
    sum(torch.cuda.get_device_properties(i).total_memory for i in range(num_gpus)) / 1e9
)
print(f"✅ GPU(s): {num_gpus}")
for i in range(num_gpus):
    print(
        f"   GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB"
    )
print(f"   Total VRAM: {total_vram:.1f} GB")
assert total_vram >= 12, f"Need at least 12GB VRAM, found {total_vram:.1f}GB"

## 2. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import os
import torch
from dotenv import load_dotenv

load_dotenv()
MY_TOKEN = os.getenv("HF_TOKEN")

MAX_SEQ_LENGTH = 2048  # Qwen 2.5 support up to 32768, 2048 saves VRAM
DTYPE = None  # Auto-detect (float16 for T4, bfloat16 for A100)
LOAD_IN_4BIT = True  # QLoRA: 4-bit base model

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-2B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    token=MY_TOKEN,
)
print("✅ Base model loaded")

## 3. Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank — 16 is good balance
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # critical for long sequences
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

# Print trainable parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("✅ LoRA adapters added")
print(
    f"   Trainable parameters: {trainable / 1e6:.1f}M / {total / 1e6:.0f}M total ({100 * trainable / total:.2f}%)"
)

## 4. Load and Prepare Dataset

In [ ]:
from datasets import load_dataset

# KAGGLE: Upload train.jsonl/val.jsonl as a dataset, then update paths
# Adjust path if running on Kaggle ('data/train.jsonl' or '/kaggle/input/.../train.jsonl')
TRAIN_PATH = "data/train.jsonl"
VAL_PATH = "data/val.jsonl"

train_ds = load_dataset("json", data_files=TRAIN_PATH, split="train")
val_ds = load_dataset("json", data_files=VAL_PATH, split="train")

print(f"Train: {len(train_ds):,} examples")
print(f"Val:   {len(val_ds):,} examples")
print("Sample:", train_ds[0])

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template='qwen-2.5',
    mapping={'role': 'role', 'content': 'content', 'user': 'user', 'assistant': 'assistant'}
)

def format_prompts(examples):
    """Apply Qwen 2.5 chat template to each example."""
    texts = []
    for messages in examples['messages']:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {'text': texts}

train_ds = train_ds.map(format_prompts, batched=True, remove_columns=train_ds.column_names)
val_ds   = val_ds.map(format_prompts, batched=True, remove_columns=val_ds.column_names)

print('✅ Chat template applied')
print('Sample text (first 200 chars):', train_ds[0]['text'][:200])

## 5. Training Configuration

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    output_dir="models/qwen_gu_health_lora",
    per_device_train_batch_size=2,  # As per roadmap
    gradient_accumulation_steps=4,  # Effective batch size = 8
    warmup_steps=50,
    num_train_epochs=2,  # As per roadmap
    learning_rate=2e-4,  # As per roadmap
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    optim="adamw_8bit",  # Memory-efficient optimizer
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("✅ Trainer configured")
print(
    f"   Training steps: {len(trainer.get_train_dataloader()) * training_args.num_train_epochs}"
)

## 6. Train!

In [ ]:
# Plot loss curve
import matplotlib.pyplot as plt
import time

print("🚀 Starting QLoRA fine-tuning...")
start = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start

print(f"\n✅ Training complete in {elapsed / 60:.1f} minutes")
print(f"   Final train loss: {trainer_stats.training_loss:.4f}")


logs = trainer.state.log_history
train_logs = [
    (log["step"], log["loss"])
    for log in logs
    if "loss" in log and "eval_loss" not in log
]
eval_logs = [(log["step"], log["eval_loss"]) for log in logs if "eval_loss" in log]

if train_logs:
    steps, losses = zip(*train_logs)
    plt.figure(figsize=(10, 4))
    plt.plot(steps, losses, label="Train Loss", color="blue")
    if eval_logs:
        eval_steps, eval_losses = zip(*eval_logs)
        plt.plot(eval_steps, eval_losses, label="Val Loss", color="orange")
    plt.xlabel("Steps")
    plt.ylabel("Loss")
    plt.title("QLoRA Training Loss — Qwen 2.5 2B (Gujarati Healthcare)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("outputs/training_loss.png", dpi=120)
    plt.show()
    print("✅ Loss curve saved to outputs/training_loss.png")

## 7. Save LoRA Adapter

In [ ]:
import os

os.makedirs("models", exist_ok=True)
ADAPTER_PATH = "models/qwen_gu_health_lora"

# Save ONLY LoRA adapter (not full model → much smaller)
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

adapter_size_mb = (
    sum(
        os.path.getsize(os.path.join(ADAPTER_PATH, f))
        for f in os.listdir(ADAPTER_PATH)
        if os.path.isfile(os.path.join(ADAPTER_PATH, f))
    )
    / 1e6
)

print(f"✅ LoRA adapter saved to {ADAPTER_PATH}")
print(f"   Adapter size: {adapter_size_mb:.1f} MB")
print(f"   Files: {os.listdir(ADAPTER_PATH)}")
print()
print("📥 DOWNLOAD the models/ folder from Kaggle Output before session ends!")
print("📝 NEXT STEP: Run Notebook 06 to build the Vector DB.")

## 8. Quick Inference Test (Post Fine-Tuning)

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)  # Enable native 2x faster inference


def test_finetuned(prompt: str) -> str:
    messages = [
        {
            "role": "system",
            "content": "You are a safe, accurate Gujarati healthcare assistant.",
        },
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    outputs = model.generate(
        inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1] :], skip_special_tokens=True)


test_prompts = [
    "ડાયાબિટીઝ ના મુખ્ય લક્ષણો ક્યા છે?",
    "ઉચ્ચ blood pressure ઘટાડવા ઘરેલૂ ઉપાય?",
    "Paracetamol ક્યારે ન લેવી?",
]

print("🧪 Post Fine-tuning Inference Test:")
print("=" * 60)
for prompt in test_prompts:
    response = test_finetuned(prompt)
    print(f"Q: {prompt}")
    print(f"A: {response}")
    print("-" * 40)